# Grid Capex-to-Alpha — U.S. Electrical Equipment
### Research-grade public-data pilot · October 19, 2026

**Question:** Does a fixed-scale gap between U.S. power-system output and electrical-equipment manufacturing output forecast equipment production six months ahead?

**Ex ante falsification standard:** the factor must beat both a momentum-only benchmark and a historical-mean benchmark, survive time-shift placebos, and be reconstructed with point-in-time vintages before any alpha claim.

**Preliminary verdict:** A structural demand case exists, but the gap factor does not beat the mean benchmark (full RMSE 4.124 pp vs mean 3.524 pp) on 19 overlapping held-out forecast origins. **No security-level returns are analysed.**

Refer to README.md for source register, hypothesis definitions, model risk, limitations and future research.

> **Notebook status — historical snapshot / not yet re-executed in this change.** The tables quoted below reproduce the *published fixed-vintage analysis* as reported in this repository. All new analysis/figure cells must be executed with the frozen CSV and pass CI before anyone treats their outputs as verified. The study is a **macro output forecasting experiment**, not stock-market alpha.


## 1 · Set-up and provenance

The included monthly government index values are a 2022-2026 official-table transcription. They are **not** an ALFRED point-in-time panel and not a data-centre load time series. Both the static source data and all execution logic are included in this directory.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from grid_alpha import read_snapshot, make_panel, analyse, walk_forward, shifted_placebo, block_bootstrap
DATA = Path('fed_industrial_production_snapshot_2022_2026.csv')
raw = read_snapshot(DATA)
raw.tail()

### 1A. Data provenance, audit boundaries and selection
The frozen source is the Federal Reserve's **G.17 industrial-production index** for utility power (IPG2211S) and electrical equipment (IPG3353S). This is not data-center consumption, announced grid CAPEX, order backlog or equity total return. Level revisions and publication-time data availability remain unresolved. The data may be useful for a preliminary falsification but cannot meet historical point-in-time security-trading standards. See `source_register.csv` for provenance and the repository README for the publication caveats.


In [ ]:
source=pd.read_csv('source_register.csv')
display(source)
assert raw.index.is_monotonic_increasing and raw.index.is_unique
assert raw.notna().all().all() and (raw[['load','equip']] > 0).all().all()
print('Snapshot:', raw.index.min().date(), raw.index.max().date(), 'n=',len(raw))


### 1B. Industry demand transmission
**Structural bull case:** incremental electrical load may require grid connections, switchgear, transformers, cable and distribution investment. **Countercase:** utility power output is affected by weather and total demand; it does not isolate AI infrastructure. The engineering and order-to-revenue lags are variable; manufacturing trade, inventories and imported equipment weaken simple domestic production mapping. Because industrial production is a **flow**, not companies' free cash flow, this notebook explicitly stops before making a security-level conclusion.


In [ ]:
fig,ax=plt.subplots(figsize=(11,4));
raw[['load','equip']].plot(ax=ax,lw=2)
ax.set_title('Official IP level indices: production, not firm earnings');ax.set_ylabel('Index');ax.grid(alpha=.25);plt.show()


### 1C. YoY behavior and revisions
Plot growth in both underlying components instead of assuming they are aligned. A same-month gap is not automatically a *leading* gap; only performance of lagged information against matured future targets can establish incremental predictability.


In [ ]:
growth=100*(raw[['load','equip']]/raw[['load','equip']].shift(12)-1)
ax=growth.plot(figsize=(11,4),title='Production growth YoY, fixed snapshot',lw=1.6)
ax.axhline(0,color='gray',linewidth=1);ax.grid(alpha=.2);ax.set_ylabel('% YoY');plt.show()


## 2 · Fixed signal and lagged feature engineering

At formation month t only month t−1 macro features are permitted. `gap = tanh((power_yoy - equip_yoy) / 10)`. Target = next-six-month percentage change in equipment IP. The parameter 10 pp is fixed before inspection; no all-sample mean/scaling enters construction.

In [ ]:
panel = make_panel(raw)
assert len(panel) == 37
panel[['gap', 'load_yoy_pit', 'equip_yoy_pit', 'y']].head()

### 2A. Prespecified factor
Let \(L_t\) and \(E_t\) be load and equipment-production indices. Define lagged growth \(g^L_{t-1}=100(L_{t-1}/L_{t-13}-1)\), \(g^E_{t-1}=100(E_{t-1}/E_{t-13}-1)\), and
\[ F_t=\tanh\left(\frac{g^L_{t-1}-g^E_{t-1}}{10}\right). \]
The scale 10 percentage points is fixed, not learned from the entire sample. The response \(Y_t=100(E_{t+6}/E_t-1)\) is **forward sector output**, not share-price performance.


In [ ]:
display(panel.describe(percentiles=[.05,.25,.5,.75,.95]).round(3))
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].hist(panel['gap'],bins=15,edgecolor='white');axs[0].set_title('Factor distribution')
axs[1].scatter(panel['gap'],panel['y'],alpha=.7);axs[1].set_xlabel('Lagged gap factor');axs[1].set_ylabel('Forward 6m output growth (%)');axs[1].set_title('Descriptive only; overlapping labels')
plt.tight_layout();plt.show()


### 2B. Publication lag versus revision leakage
Moving features back one calendar month is **only a publication-buffer assumption**. Even if each observation was released before formation, a value downloaded in October 2026 can contain later benchmark revisions. The correct production implementation would join every historical forecast date to a separately archived vintage and respect its actual release timestamp.


## 3 · Expanding OOS predictions and baselines

The training set for an origin t is restricted to labels with formation dates no later than t−7 months (six-month target maturity + one publication month). A train-only standardized Ridge with equipment growth/momentum is compared with the same model + gap and a simple expanding historical mean. These are **economic forecasts**, not position returns.

In [ ]:
results, panel, oos = analyse(DATA)
import json
print(json.dumps(results, indent=2, ensure_ascii=False))
oos.tail(10)

### 3A. Model-identical scorecard
The historical mean, equipment momentum controls, and controls + gap must share **exactly the same forecast origins**. Train the Ridge scalers only on the expanding training fold. The original implementation purges training origins until the target has matured by six months plus an additional month. Small sample caution: only **19** overlapping out-of-sample origins.


In [ ]:
metrics=pd.DataFrame(results['forecast_metrics']).T
metrics.index.name='Model';display(metrics.round(4))
assert len(oos)==results['oos_forecast_origins']
assert set(['mean','baseline','with_gap']).issubset(oos.columns)


The reported snapshot has mean RMSE **3.524 pp**, equipment-control Ridge **4.228 pp**, and factor-enhanced Ridge **4.124 pp**. The gap helps slightly *relative to the control*, but **fails the simplest historical-mean hurdle**; the research decision is therefore negative.


In [ ]:
e=oos[['mean','baseline','with_gap']].sub(oos['actual'],axis=0)
fig,axes=plt.subplots(1,2,figsize=(13,4))
for name in ['mean','baseline','with_gap']:
 axes[0].plot(e.index,e[name].abs(),label=name,alpha=.85)
axes[0].legend();axes[0].set_title('Absolute forecast error by origin');axes[0].set_ylabel('pp')
axes[1].boxplot([e[c] for c in e],tick_labels=list(e.columns));axes[1].set_title('Signed forecast error distribution');axes[1].axhline(0,color='gray',lw=1)
plt.tight_layout();plt.show()


### 3B. Loss differential and false precision
The relevant loss comparison is \(d_t=(Y_t-\hat Y^{base}_t)^2-(Y_t-\hat Y^{full}_t)^2\). Positive values favor the augmented factor. Overlapping six-month horizons make adjacent \(d_t\) correlated; a conventional independent-sample t-test would be misleading.


In [ ]:
loss_gain=(oos.actual-oos.baseline)**2-(oos.actual-oos.with_gap)**2
fig,ax=plt.subplots(figsize=(10,3.5))
ax.bar(oos.index,loss_gain,width=20);ax.axhline(0,color='gray',lw=1)
ax.set_title('Paired squared-error gain: control minus full');ax.set_ylabel('pp²');plt.show()
print('Mean paired squared-error gain:',round(float(loss_gain.mean()),4))


## 4 · Placebo, dependence and bias diagnostics

Circularly shift the observed gap series by 7+ periods and refit the *entire* walk-forward sequence. This checks some alignment fragility but is **not** an exact time-series randomization test. Additionally resample six-month forecast errors in blocks; the tiny and overlapping OOS sample is not adequate for credible confidence bounds.

In [ ]:
print('Placebo:', results['circular_shift_placebo'])
print('Block-bootstrap illustrative interval:', results['block_bootstrap_mse_improvement_95pct_exploratory'])
print('Rank correlation (nominal only):', results['full_sample_spearman'], results['naive_nominal_spearman_p'])

### 4A. Placebo protocol and what it cannot prove
The circular shifts preserve some autocorrelation in the factor series but alter its alignment relative to equipment output. This is an *exploratory* alignment diagnostic, not a valid exact randomization test for a nonstationary, autocorrelated economic series. The recorded exploratory p-value is approximately **0.32**, and should not be described as significant.


In [ ]:
pl=results['circular_shift_placebo'];ci=results['block_bootstrap_mse_improvement_95pct_exploratory']
display(pd.DataFrame([{'shifts':pl.get('n_shifts'), 'null_median_rmse':pl.get('null_median_rmse'),'exploratory_p':pl.get('placebo_p'), 'block_95pct_low':ci[0], 'block_95pct_high':ci[1]}]).round(4))


### 4B. Limited-sample fragility
Nineteen monthly forecast origins with a six-month horizon are far from 19 independent observations. Display the training-sample growth over time and inspect how a handful of bad forecast months drive the result. A rolling historical split is not enough to establish a business-cycle-generalizable investment edge.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,3.5))
oos['n_train'].plot(ax=axes[0],title='Information set expands')
oos[['actual','mean','with_gap']].plot(ax=axes[1],title='Outcome and challenger vs naive')
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout();plt.show()


## 5 · Visual audit

Latest-revised production indices can co-move in ways unrelated to AI. Display outcomes and forecast errors; avoid presenting a cumulative P&L for data that do not include investable returns.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
raw[['load', 'equip']].plot(ax=axes[0], title='Federal Reserve industrial-production indices (latest/mixed vintages)')
oos[['actual', 'mean', 'baseline', 'with_gap']].plot(ax=axes[1], title='Next-six-month equipment output growth, percentage points')
plt.tight_layout()
plt.show()

### 5A. Bias register — explicit pass/fail
| Risk | Finding | Gate |
|---|---|---|
| Calendar look-ahead | one-month feature lag and matured labels | Partial |
| Historical revisions | no archived G.17 vintages | **FAIL** |
| Data-center causality | utility IP conflates all power demand | **FAIL** |
| Sector exposure | no dated company/product weights | **FAIL** |
| Survivorship | no contemporaneous securities universe | Not tested |
| Overlapping targets | six-month labels; 19 OOS origins | High uncertainty |
| Multiple testing | exploratory choice of industry and gap | Unresolved |
| Trading costs | no tradable prices or turnover | Not tested |


### 5B. Concrete security-level experiment
Freeze a public universe of grid-equipment suppliers at each formation date; extract prior-filed revenue by transformers, switchgear, power equipment and geographic end-market, and weight firm signals only using the then-published mix. Align to industrial-series release dates and model next-quarter *reported* orders/margins and subsequent **industry-adjusted, split/dividend/delisting-adjusted total returns**. Compare rank IC and top/bottom portfolios to size, value, momentum and earnings-revision controls; account for effective transaction costs, turnover, delistings and multiple comparisons.


## 6 · Equity alpha stage gate — NOT PASSED

Necessary data not present: contemporaneous security universes; adjusted total returns and dividend/delisting information; release-known stock-to-industry exposure weights; regional interconnection and load data; financing/trading costs. **Do not convert this macro prediction exercise into a claim of stock-market alpha.**

A future version should test 10-K-observable exposures against next-quarter earnings revisions, sector-neutral rank IC and fully adjusted portfolio returns under historical vintage constraints. See README.md §§7–9.

## 7 · Final verdict and replication gate
**Outcome:** plausible secular electrical-equipment opportunity; **no robust macro predictive alpha** and **no investable equity-alpha result**. The full model loses to the historical mean, and its exploratory placebo does not support a strong claim. All additional display/figure cells in this amended notebook are **unexecuted until run-all/CI verifies them**. Reproduce from this folder with `pip install -r requirements.txt`, `python -m pytest -q test_grid_alpha.py`, then `jupyter nbconvert --to notebook --execute grid_capex_alpha_research.ipynb --output executed.ipynb`. Never replace the original fixed snapshot without preserving its retrieval date.
